# Metal Defect Classification
This notebook trains a CNN model to classify metal defects using the NEU-DET dataset.

In [ ]:
import os
import glob
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm import tqdm
import pickle

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## Dataset Loading

In [ ]:
# Define the dataset
class MetalDefectDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.image_paths = glob.glob(os.path.join(root_dir, 'images', '*.jpg'))
        self.transform = transform
        self.classes = ['crazing', 'inclusion', 'patches', 'pitted_surface', 'rolled-in_scale', 'scratches']
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
    def __len__(self):
        return len(self.image_paths)
        
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        
        # Extract label from filename (e.g., crazing_1.jpg -> crazing)
        filename = os.path.basename(img_path)
        
        if "rolled-in_scale" in filename:
            label_name = "rolled-in_scale"
        elif "pitted_surface" in filename:
            label_name = "pitted_surface"
        else:
            label_name = filename.split('_')[0]
            
        label = self.class_to_idx[label_name]
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

# Define transforms
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Paths (Updated to absolute paths as per data structure)
train_dir = r'E:\Semester 8\Advanced Methods\Project\data\NEU-DET\train'
val_dir = r'E:\Semester 8\Advanced Methods\Project\data\NEU-DET\validation'

train_dataset = MetalDefectDataset(train_dir, transform=transform)
val_dataset = MetalDefectDataset(val_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")

## Model Architecture Definition

In [ ]:
# Define Simple CNN Model
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(SimpleCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 112x112
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2), # 56x56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2, 2)  # 28x28
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

## Training Loop
We train the model, show progress with remaining time using tqdm, and calculate accuracy. If accuracy improves, we save it as a pickle file (`metal_defect_model_notebook.pkl`).

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10):
    best_acc = 0.0
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        # tqdm for progress bar showing percent and remaining time
        progress_bar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{num_epochs}', leave=True)
        
        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            progress_bar.set_postfix({'loss': running_loss / (progress_bar.n + 1)})
            
        # Validation accuracy
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs.data, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        val_acc = 100 * correct / total
        print(f'Validation Accuracy: {val_acc:.2f}%')
        
        if val_acc > best_acc:
            best_acc = val_acc
            # Save the best model using pickle-based torch.save
            torch.save(model.state_dict(), 'metal_defect_model_notebook.pkl')
            print("Model saved to metal_defect_model_notebook.pkl (Accuracy improved)")
            
    return best_acc

print("Starting training...")
best_accuracy = train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=10)
print(f"Training completed. Best Validation Accuracy: {best_accuracy:.2f}%")